# In-Class Assignment: Teach a CNN Your Own Categories

**Team members:** *write your names here*

You will teach a CNN to recognize **2 or 3 categories of images that you choose**, using **transfer learning**: we borrow the "find features" part (Part 1) of a CNN that was already trained on 1.2 million photos, and train only a small "classify" part (Part 2) on your images.

**Before you start (in Colab):** `Runtime → Change runtime type → T4 GPU` (faster, but the CPU also works). Then run the cells from top to bottom.

### Step 0. Collect your images (about 10 minutes)

1. Choose **2 or 3 categories**, for example *cats vs dogs*, *pizza vs burger vs taco*, *sedan vs SUV vs truck*, or *Bryant buildings*.
2. Download **about 25 images per category** (Google Images works; the Imageye Chrome extension saves many images at once). Any of jpg, png, webp, jfif is fine.
3. Put them in **one folder per category**, put those folders in one folder, and **zip it**:

```
my_images.zip
    cats/   img1.jpg, img2.png, ...
    dogs/   ...
```

### Step 1. Upload your zip file

Run the cell and choose your zip file. The cell unzips it, finds your category folders, and converts every image to `.jpg` (images that cannot be opened are skipped).

In [ ]:
import os, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ZIP_FILE = None   # leave as None: you will be asked to choose the file
if ZIP_FILE is None:
    from google.colab import files
    ZIP_FILE = list(files.upload())[0]

shutil.rmtree("unzipped", ignore_errors=True)
with zipfile.ZipFile(ZIP_FILE) as z:
    z.extractall("unzipped")

def is_category_parent(folder):
    """True if the folder holds 2 or more subfolders (the categories)."""
    subs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)) and not d.startswith((".", "__"))]
    return len(subs) >= 2

root = next(r for r, d, f in os.walk("unzipped") if "__MACOSX" not in r and is_category_parent(r))

shutil.rmtree("images", ignore_errors=True)
for cat in sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)) and not d.startswith((".", "__"))):
    os.makedirs(f"images/{cat}")
    kept = 0
    for f in sorted(os.listdir(os.path.join(root, cat))):
        try:
            Image.open(os.path.join(root, cat, f)).convert("RGB").save(f"images/{cat}/{kept:04d}.jpg")
            kept += 1
        except Exception:
            pass   # not an image
    print(f"{cat}: {kept} images")

### Step 2. Load the images

20% of your images are held out as a **validation set**: the model never trains on them, so they tell us how well it works on new images.

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    "images", validation_split=0.2, subset="both", seed=1, image_size=IMG_SIZE, batch_size=16)
class_names = train_ds.class_names
print("Categories:", class_names)

images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 3))
for i in range(min(8, len(images))):
    plt.subplot(1, 8, i + 1)
    plt.imshow(images[i].numpy().astype("uint8")); plt.title(class_names[labels[i]], fontsize=9); plt.axis("off")
plt.show()

### Step 3. Build the CNN: Part 1 borrowed, Part 2 new

- **Part 1 (find features):** MobileNetV2, a CNN trained on 1.2 million photos (ImageNet). We **freeze** it: its filters are not changed.
- **Part 2 (classify):** average each feature map into one number, then one dense layer with softmax for **your** categories. Only this part is trained.

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False                                            # freeze Part 1

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.layers.RandomFlip("horizontal")(inputs)              # small random changes: more variety for training
x = tf.keras.layers.RandomRotation(0.1)(x)
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(x)            # pixels 0..255 -> -1..1, as MobileNetV2 expects
x = base(x, training=False)                                       # Part 1: find features
x = tf.keras.layers.GlobalAveragePooling2D()(x)                   # Part 2: one number per feature map ...
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)   # ... then dense + softmax
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

total = model.count_params()
trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print(f"Total parameters:     {total:,}")
print(f"Trained on your data: {trainable:,}")

### Step 4. Train

10 passes over your images (about 1 minute on a GPU, a few minutes on a CPU).

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10, verbose=2)

plt.plot(history.history["accuracy"], label="training")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1.05); plt.legend(); plt.show()
print(f"Validation accuracy: {history.history['val_accuracy'][-1]:.2f}   (random guessing: {1 / len(class_names):.2f})")

### Step 5. Look at the validation images

Each validation image with the model's prediction and confidence. **Red titles are mistakes.**

In [ ]:
val_images, val_labels = [], []
for xb, yb in val_ds:
    val_images.append(xb.numpy()); val_labels.append(yb.numpy())
val_images, val_labels = np.concatenate(val_images), np.concatenate(val_labels)
proba = model.predict(val_images, verbose=0)
pred = proba.argmax(1)

n_show = min(18, len(val_images))
order = np.argsort(pred == val_labels, kind="stable")[:n_show]    # mistakes first
plt.figure(figsize=(15, 2.8 * ((n_show + 5) // 6)))
for k, i in enumerate(order):
    plt.subplot((n_show + 5) // 6, 6, k + 1)
    plt.imshow(val_images[i].astype("uint8")); plt.axis("off")
    ok = pred[i] == val_labels[i]
    plt.title(f"true: {class_names[val_labels[i]]}\npred: {class_names[pred[i]]} ({proba[i].max():.0%})",
              fontsize=9, color="black" if ok else "red")
plt.tight_layout(); plt.show()
print(f"Mistakes: {(pred != val_labels).sum()} of {len(val_labels)}")

### Step 6. Stump the Model 🏆

Find **new** images (not in your zip) that you think will **fool** your model: a strange angle, a cartoon, a look-alike, a busy background... Upload 1 to 5 of them. The best "fool" (a **wrong** answer with **high confidence**) will be shown to the class at the end!

In [ ]:
STUMP_FILES = None   # leave as None: you will be asked to choose the files
if STUMP_FILES is None:
    from google.colab import files
    STUMP_FILES = list(files.upload())

plt.figure(figsize=(3.2 * len(STUMP_FILES), 3.4))
for k, f in enumerate(STUMP_FILES):
    img = Image.open(f).convert("RGB")
    p = model.predict(np.array(img.resize(IMG_SIZE), dtype="float32")[None], verbose=0)[0]
    plt.subplot(1, len(STUMP_FILES), k + 1)
    plt.imshow(img); plt.axis("off")
    plt.title(f"{class_names[p.argmax()]} ({p.max():.0%})", fontsize=11)
plt.show()

## Questions

Answer in a few sentences each, **about your own results**.

**Question 1 (the two parts).** Step 3 printed the total number of parameters and the number trained on your data. Which part of the network (Part 1 or Part 2) do the trained parameters belong to? Why can we get a good model from only about 25 images per category?

**Your answer:** *Write your answer here.*

**Question 2 (your accuracy).** What is your validation accuracy, and how does it compare with random guessing? Look at the training and validation curves: do you see overfitting?

**Your answer:** *Write your answer here.*

**Question 3 (a mistake).** Pick one red (wrong) image from Step 5, or the least confident one if there are no mistakes. What in that image could have confused the model?

**Your answer:** *Write your answer here.*

**Question 4 (stump the model).** Which of your images fooled the model, and with what confidence? Why do you think it worked? If nothing fooled it, which image came closest?

**Your answer:** *Write your answer here.*